In [1]:
!pip install transformers==4.40.0 torch==2.3.0 datasets==2.19.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 2.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 43.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 719.7 kB/s eta 0:00:000:01m00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 26.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 47.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 12.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import torch
print(torch.cuda.is_available())

True


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [4]:
device= 'cuda' if torch.cuda.is_available() else 'cpu'

draft_model = AutoModelForCausalLM.from_pretrained(
    'gpt2',
    torch_dtype = torch.float16
).to(device).eval()



/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [5]:
tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
target_model = AutoModelForCausalLM.from_pretrained(
    'gpt2-xl',
    torch_dtype = torch.float16
).to(device).eval()

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [7]:
print(torch.cuda.is_available())

True


In [8]:
from tokenizers import Tokenizer
import time
def autoregressive_generate(model, input_ids, max_new_tokens=50, temperature=1.0):
  generated = input_ids.clone()
  with torch.no_grad():
    for _ in range(max_new_tokens):
      output = model(generated)
      logits = output.logits[:, -1, :] / temperature
      probs = torch.softmax(logits, dim=-1)
      next_token = torch.multinomial(probs, num_samples=1)
      generated = torch.cat([generated, next_token], dim=-1)
      if next_token.item() == tokenizer.eos_token_id:
        break
  return generated

prompt = "Ai is the technology that will "
input_ids=tokenizer(prompt, return_tensors='pt')['input_ids'].to(device)
to = time.time()
N_RUNS = 5
for _ in range(N_RUNS):
  output = autoregressive_generate(target_model, input_ids, max_new_tokens=50)
t_baseline = (time.time()-to)/N_RUNS
print(f"Baseline Time - AVG {t_baseline:.3f}")
print(f"Tokens per sec - {50/t_baseline:.1f}")


Baseline Time - AVG 2.455
Tokens per sec - 20.4


In [ ]:
def generate_draft_token(draft_model, input_ids, k=6, temperature=1.0):
  generated = input_ids.clone()
  token_output = []
  output_prob = []
  with torch.no_grad():
    for _ in range(k):
      output = draft_model(generated)
      logits = output.logits[:, -1, :] / temperature
      probs = torch.softmax(logits, dim=-1)
      next_token = torch.multinomial(probs, num_samples=1)
      generated = torch.cat([generated, next_token], dim=-1)
      token_output.append(next_token)
      output_prob.append(probs)
      #generated = torch.cat([generated, next_token], dim=-1)
      if next_token.item() == tokenizer.eos_token_id:
        break
  return generated, token_output, output_prob


k = 4
draft_seq, draft_output , draft_prob = generate_draft_token(draft_model, input_ids,k=k)
print(f"Draft Sequence - {tokenizer.decode(draft_seq[0])}")
#print(f"Draft Output - {tokenizer.decode(draft_output[0])}")
print(f"Draft Prob Shape - {draft_prob[0].shape}")


SyntaxError: invalid syntax (3805003479.py, line 3)